In [1]:
import pandas as pd
import sys
from pathlib import Path
root_path = Path().resolve().parent

sys.path.append(str(root_path))
from utils.cleaning import *
from utils.feature_engineering import *
from utils.transformer import *
from utils.encoding import *
from utils.scaling import *
from utils.feature_selection import *
from utils.validation import pipeline_report

from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv("data/winequality-red.csv")

### Data Cleaning

In [3]:
df = handle_whitespace(df)

[handle_whitespace] Stripped whitespace on 0 columns


In [4]:
df = drop_duplicates(df)

[drop_duplicates] Removed 240 duplicate rows (1599 → 1359)


### Impute Outlier

In [5]:
X = df.drop(columns=['quality'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from utils.eda import outlier_report

outlier_report(X_train)

OUTLIER REPORT  (method=iqr)
              column  n_outliers  pct_outliers  has_outlier  lower_bound  upper_bound
      residual_sugar          94          8.65         True       0.8500       3.6500
           chlorides          75          6.90         True       0.0385       0.1225
           sulphates          46          4.23         True       0.2800       1.0000
       fixed_acidity          39          3.59         True       4.1000      12.1000
total_sulfur_dioxide          36          3.31         True     -39.5000     124.5000
             density          28          2.58         True       0.9923       1.0011
                  pH          22          2.02         True       2.9250       3.6850
 free_sulfur_dioxide          20          1.84         True     -14.0000      42.0000
    volatile_acidity          12          1.10         True       0.0150       1.0150
             alcohol          11          1.01         True       7.1000      13.5000
         citric_acid     

In [7]:
X_train, X_test = cap_outliers(X_train, X_test, X_train.columns.tolist())

[cap_outliers] 'fixed_acidity': Clipped to IQR bounds [4.1000, 12.1000]
[cap_outliers] 'volatile_acidity': Clipped to IQR bounds [0.0150, 1.0150]
[cap_outliers] 'citric_acid': Clipped to IQR bounds [-0.3800, 0.9000]
[cap_outliers] 'residual_sugar': Clipped to IQR bounds [0.8500, 3.6500]
[cap_outliers] 'chlorides': Clipped to IQR bounds [0.0385, 0.1225]
[cap_outliers] 'free_sulfur_dioxide': Clipped to IQR bounds [-14.0000, 42.0000]
[cap_outliers] 'total_sulfur_dioxide': Clipped to IQR bounds [-39.5000, 124.5000]
[cap_outliers] 'density': Clipped to IQR bounds [0.9923, 1.0011]
[cap_outliers] 'pH': Clipped to IQR bounds [2.9250, 3.6850]
[cap_outliers] 'sulphates': Clipped to IQR bounds [0.2800, 1.0000]
[cap_outliers] 'alcohol': Clipped to IQR bounds [7.1000, 13.5000]


### Feature Scaling

In [8]:
outlier_report(X_train)

OUTLIER REPORT  (method=iqr)
Empty DataFrame
Columns: [column, n_outliers, pct_outliers, has_outlier, lower_bound, upper_bound]
Index: []

Kolom dengan outlier: 0 / 11


In [9]:
from utils.scaling import standardize
X_train, X_test, scaler = standardize(X_train, X_test, X_train.columns.tolist())

[standardize] Standardized 11 columns.


In [10]:
print(pipeline_report(X_train))
print(pipeline_report(X_test))

  PIPELINE REPORT

  Shape        : 1087 rows × 11 columns
  Dtypes       : float64: 11
  Missing      : 0 cells (0.00%) across 0 columns
  Duplicates   : 1 rows (0.09%)
  Outlier cols  : 6 / 11 columns have outliers
None
  PIPELINE REPORT

  Shape        : 272 rows × 11 columns
  Dtypes       : float64: 11
  Missing      : 0 cells (0.00%) across 0 columns
  Duplicates   : 0 rows (0.00%)
  Outlier cols  : 5 / 11 columns have outliers
None


In [11]:
pipeline_report(X_test, target=None)

  PIPELINE REPORT

  Shape        : 272 rows × 11 columns
  Dtypes       : float64: 11
  Missing      : 0 cells (0.00%) across 0 columns
  Duplicates   : 0 rows (0.00%)
  Outlier cols  : 5 / 11 columns have outliers


In [12]:
print(y_test.isna().sum())
print(y_train.isna().sum())

0
0


In [13]:
import json
import joblib
from pathlib import Path

Path('data').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

# export scaler & IQR bounds untuk inference nanti
joblib.dump(scaler, 'models/scaler.pkl')
iqr_bounds = {}
for col in X_train.columns:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_bounds[col] = {
        "lower": round(Q1 - 1.5 * IQR, 4),
        "upper": round(Q3 + 1.5 * IQR, 4),
    }
with open('models/iqr_bounds.json', 'w') as f:
    json.dump(iqr_bounds, f, indent=2)

print(f"[Export] Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print("Files: data/X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Artifacts: models/scaler.pkl, models/iqr_bounds.json")

[Export] Train: 1087 rows, Test: 272 rows
Files: data/X_train.csv, X_test.csv, y_train.csv, y_test.csv
Artifacts: models/scaler.pkl, models/iqr_bounds.json
